# Simplicio 27B · Fusão do LoRA no modelo base (merge → 16-bit → Hugging Face)
1. Runtime > Change runtime type > **A100 GPU** (high-RAM se disponível).
2. Run all. Na célula 3 cole um **HF token com permissão de escrita**.
Saída: tudo no repo único `wesleysimplicio/Simplicio-27B` (adapter, merge 16-bit e GGUF).


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

In [ ]:
!pip install -q unsloth
!pip install -q --upgrade huggingface_hub

In [ ]:
from getpass import getpass
import os
os.environ['HF_TOKEN'] = getpass('HF token (write): ')
from huggingface_hub import login; login(os.environ['HF_TOKEN'])

In [ ]:
# Carrega base (4-bit, igual ao treino) + adapter LoRA Simplicio-27B
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="wesleysimplicio/Simplicio-27B",  # adapter; unsloth resolve a base via adapter_config
    max_seq_length=16384, load_in_4bit=True)
print(type(model))

In [ ]:
# Teste rápido antes de fundir
FastLanguageModel.for_inference(model)
msgs=[{"role":"user","content":"Escreva um diff SEARCH/REPLACE que renomeia a função foo para bar em utils.py"}]
ids=tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
out=model.generate(ids, max_new_tokens=200)
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

In [ ]:
# FUSÃO: dequantiza a base para 16-bit, soma o LoRA e publica o modelo completo
REPO="wesleysimplicio/Simplicio-27B"
model.push_to_hub_merged(REPO, tokenizer, save_method="merged_16bit", token=os.environ['HF_TOKEN'])
print("Publicado:", "https://huggingface.co/"+REPO)


In [ ]:
# (Opcional) GGUF Q4_K_M para Ollama/llama.cpp
GGUF="wesleysimplicio/Simplicio-27B"
model.push_to_hub_gguf(GGUF, tokenizer, quantization_method="q4_k_m", token=os.environ['HF_TOKEN'])
print("Publicado:", "https://huggingface.co/"+GGUF)
